# Cell 1 — Install + Verify

In [1]:
import sys, subprocess

def ensure(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", pkg])

# Install only if missing
try:
    import segmentation_models_pytorch as smp
except Exception:
    ensure("segmentation-models-pytorch")
    ensure("timm")

import torch
import segmentation_models_pytorch as smp

print("torch:", torch.__version__)
print("smp:", smp.__version__)
print("cuda available:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 30.7 MB/s eta 0:00:00
torch: 2.10.0+cu128
smp: 0.5.0
cuda available: True


# Cell 2 — Imports

In [2]:
import os, glob, json, math, hashlib, random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2

from torch.utils.data import Dataset, DataLoader

# Cell 3 — Config + Paths (auto-detect your files)

In [3]:
# ---- Config ----
SEED = 7
THR = 0.5
BATCH_SIZE = 12
NUM_WORKERS = 2
EVAL_SIZES = [256, 448]

OUT_ROOT = Path("/kaggle/working/output/phase7_model1")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(SEED)

# ---- Find splits ----
val_csv = glob.glob("/kaggle/input/**/val_split.csv", recursive=True)[0]
train_csv = glob.glob("/kaggle/input/**/train_split.csv", recursive=True)[0]
print("val_csv:", val_csv)
print("train_csv:", train_csv)

val_df = pd.read_csv(val_csv)
print("val columns:", val_df.columns.tolist())
print("val rows:", len(val_df))

# ---- Find checkpoint candidates ----
ckpt_cands = sorted(glob.glob("/kaggle/input/**/*unet*efficientnetb3*best*.pt", recursive=True))
print("\nCheckpoint candidates:")
for p in ckpt_cands:
    print(" -", p)

def pick_model1_ckpt(cands):
    # Prefer Model-1 folder
    for p in cands:
        if "model-1-checkpoints" in p.lower():
            return p
    # Prefer your naming style
    for p in cands:
        if "unet_efficientnetb3_best_ep" in os.path.basename(p).lower():
            return p
    return cands[0] if cands else None

MODEL1_CKPT = pick_model1_ckpt(ckpt_cands)
if MODEL1_CKPT is None:
    raise FileNotFoundError("Could not find Model-1 checkpoint under /kaggle/input")

print("\n Using MODEL1_CKPT:", MODEL1_CKPT)

val_csv: /kaggle/input/datasets/arman00p/phase-3-csv-files/val_split.csv
train_csv: /kaggle/input/datasets/arman00p/phase-3-csv-files/train_split.csv
val columns: ['stem', 'image_path', 'mask_path', 'source', 'empty_mask', 'has_crack']
val rows: 1695

Checkpoint candidates:
 - /kaggle/input/datasets/arman00p/model-1-checkpoints-unet-efficientnet-b3/output/SAVED_MODELS/unet_efficientnetb3_best_ep029_20260219_205815.pt
 - /kaggle/input/datasets/arman00p/model-3-checkpoints-datasetunet-eff-net-b3/unetplusplus_efficientnetb3_best.pt

 Using MODEL1_CKPT: /kaggle/input/datasets/arman00p/model-1-checkpoints-unet-efficientnet-b3/output/SAVED_MODELS/unet_efficientnetb3_best_ep029_20260219_205815.pt


# Cell 4 — Device Selection (prevents your CUDA crash)

In [4]:
def choose_device():
    if not torch.cuda.is_available():
        return torch.device("cpu")

    # If GPU is too old for current torch build, CUDA will crash.
    # Kaggle P100 is sm_60 (capability 6.0). Many recent wheels drop it.
    cap = torch.cuda.get_device_capability(0)  # (major, minor)
    name = torch.cuda.get_device_name(0)
    print("GPU:", name, "| capability:", cap)

    if cap[0] < 7:
        print("⚠️ GPU capability < 7.0 -> forcing CPU to avoid CUDA 'no kernel image' crash.")
        return torch.device("cpu")

    return torch.device("cuda")

device = choose_device()
print(" Using device:", device)

GPU: Tesla P100-PCIE-16GB | capability: (6, 0)
⚠️ GPU capability < 7.0 -> forcing CPU to avoid CUDA 'no kernel image' crash.
 Using device: cpu


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


# Cell 5 — Robust checkpoint extraction + load model

In [5]:
def extract_state_dict_any(ckpt_obj):
    """
    Robustly locate the state_dict inside many possible checkpoint formats.
    Strategy:
      1) If dict of tensors -> it is state_dict
      2) If known keys exist (state_dict/model_state_dict/model/net/weights) -> use them
      3) Else recursively search for the largest dict whose values are tensors
    """
    import torch

    # 1) Already a pure state_dict?
    if isinstance(ckpt_obj, dict) and len(ckpt_obj) > 0:
        if all(isinstance(v, torch.Tensor) for v in ckpt_obj.values()):
            return ckpt_obj

    # 2) Known container keys
    if isinstance(ckpt_obj, dict):
        for key in ["state_dict", "model_state_dict", "model", "net", "weights"]:
            if key in ckpt_obj and isinstance(ckpt_obj[key], dict):
                sd = ckpt_obj[key]
                if len(sd) > 0 and all(isinstance(v, torch.Tensor) for v in sd.values()):
                    return sd

    # 3) Recursive search for biggest tensor dict
    best = None

    def walk(obj):
        nonlocal best
        if isinstance(obj, dict):
            # candidate
            if len(obj) > 0 and all(isinstance(v, torch.Tensor) for v in obj.values()):
                if best is None or len(obj) > len(best):
                    best = obj
            # recurse
            for v in obj.values():
                walk(v)
        elif isinstance(obj, (list, tuple)):
            for v in obj:
                walk(v)

    walk(ckpt_obj)
    if best is not None:
        return best

    raise ValueError("Could not locate a tensor state_dict inside this checkpoint.")

def strip_prefixes(sd):
    cleaned = {}
    for k, v in sd.items():
        nk = k
        for pref in ("module.", "model.", "net."):
            if nk.startswith(pref):
                nk = nk[len(pref):]
        cleaned[nk] = v
    return cleaned

# Load checkpoint
ckpt = torch.load(MODEL1_CKPT, map_location="cpu")
sd = extract_state_dict_any(ckpt)
sd = strip_prefixes(sd)
print(" Extracted state_dict keys:", len(sd))

# Try both encoder families
candidates = ["efficientnet-b3", "timm-efficientnet-b3"]
best = None

for enc in candidates:
    m = smp.Unet(encoder_name=enc, encoder_weights=None, in_channels=3, classes=1, activation=None)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    score = len(missing) + len(unexpected)
    print(f"[{enc}] missing={len(missing)} unexpected={len(unexpected)} score={score}")
    if best is None or score < best["score"]:
        best = {"encoder": enc, "model": m, "missing": missing, "unexpected": unexpected, "score": score}

chosen_encoder = best["encoder"]
model = best["model"].to(device).eval()

print("\n Chosen encoder:", chosen_encoder, "| score:", best["score"])

# Hard guard: if score is huge, stop (means wrong weights / wrong architecture)
if best["score"] > 150:
    print("\n🚨 Too many missing/unexpected keys. This usually means wrong checkpoint format or mismatch.")
    print("Example missing keys:", best["missing"][:15])
    print("Example unexpected keys:", best["unexpected"][:15])
    raise RuntimeError("Checkpoint did not load correctly. Stop here and fix before evaluating.")

 Extracted state_dict keys: 634
[efficientnet-b3] missing=494 unexpected=572 score=1066
[timm-efficientnet-b3] missing=0 unexpected=0 score=0

 Chosen encoder: timm-efficientnet-b3 | score: 0


# Cell 6 — Dataset / DataLoader

In [6]:
class CrackValDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_size: int, preprocess_fn):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.preprocess_fn = preprocess_fn

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = str(row["image_path"])
        msk_path = str(row["mask_path"])

        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA).astype(np.float32)

        m = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            raise FileNotFoundError(msk_path)
        m = cv2.resize(m, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        m = (m > 127).astype(np.uint8)  # 0/1

        img = self.preprocess_fn(img)  # encoder-consistent preprocessing

        img_t = torch.from_numpy(img).permute(2, 0, 1).float()
        m_t = torch.from_numpy(m).unsqueeze(0).float()

        meta = {
            "stem": row.get("stem", ""),
            "source": row.get("source", ""),
            "empty_mask": int(row.get("empty_mask", int(m.sum() == 0))),
            "has_crack": int(row.get("has_crack", int(m.sum() > 0))),
            "image_path": img_path,
            "mask_path": msk_path,
        }
        return img_t, m_t, meta

# Cell 7 — Metrics (empty-aware IoU/Dice + pixel accuracy)

In [7]:
def iou_empty_aware(gt, pr, eps=1e-7):
    gt = gt.astype(bool)
    pr = pr.astype(bool)
    inter = np.logical_and(gt, pr).sum()
    union = np.logical_or(gt, pr).sum()
    if union == 0:
        return 1.0
    return float(inter / (union + eps))

def dice_empty_aware(gt, pr, eps=1e-7):
    gt = gt.astype(bool)
    pr = pr.astype(bool)
    inter = np.logical_and(gt, pr).sum()
    denom = gt.sum() + pr.sum()
    if denom == 0:
        return 1.0
    return float((2 * inter) / (denom + eps))

def pixel_acc(gt, pr):
    return float((gt == pr).mean())

# Cell 8 — Evaluate one resolution (writes CSVs)

In [8]:
def eval_one_size(img_size: int):
    run_name = f"unet_efficientnetb3_eval{img_size}"
    out_dir = OUT_ROOT / run_name
    out_dir.mkdir(parents=True, exist_ok=True)

    preprocess_fn = smp.encoders.get_preprocessing_fn(chosen_encoder, pretrained="imagenet")
    ds = CrackValDataset(val_df, img_size=img_size, preprocess_fn=preprocess_fn)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    rows = []

    for imgs, msks, metas in dl:
        imgs = imgs.to(device)
        with torch.no_grad():
            logits = model(imgs)
            prob = torch.sigmoid(logits).cpu().numpy()[:, 0]

        gt = msks.numpy()[:, 0].astype(np.uint8)
        pr = (prob >= THR).astype(np.uint8)

        for i in range(prob.shape[0]):
            gt_i = gt[i]
            pr_i = pr[i]
            prob_i = prob[i]

            gt_pixels = int(gt_i.sum())
            pr_pixels = int(pr_i.sum())
            has_crack_gt = int(gt_pixels > 0)
            pred_any = int(pr_pixels > 0)

            iou = iou_empty_aware(gt_i, pr_i)
            dice = dice_empty_aware(gt_i, pr_i)
            acc = pixel_acc(gt_i, pr_i)

            if pr_pixels > 0:
                pred_probs = prob_i[pr_i == 1]
                conf_mean = float(pred_probs.mean())
                conf_p95 = float(np.percentile(pred_probs, 95))
            else:
                conf_mean = 0.0
                conf_p95 = 0.0

            meta = {k: metas[k][i] for k in metas.keys()}
            rows.append({
                "stem": meta.get("stem", ""),
                "source": meta.get("source", ""),
                "image_path": meta.get("image_path", ""),
                "mask_path": meta.get("mask_path", ""),
                "empty_mask": int(meta.get("empty_mask", int(gt_pixels == 0))),
                "has_crack": int(meta.get("has_crack", has_crack_gt)),
                "has_crack_gt": has_crack_gt,
                "pred_any": pred_any,
                "gt_pixels": gt_pixels,
                "pred_pixels": pr_pixels,
                "iou_empty_aware": iou,
                "dice_empty_aware": dice,
                "pixel_acc": acc,
                "conf_mean_pred": conf_mean,
                "conf_p95_pred": conf_p95,
            })

    metrics_df = pd.DataFrame(rows)
    metrics_df.to_csv(out_dir / f"{run_name}_val_metrics.csv", index=False)

    n_total = len(metrics_df)
    crack_only = metrics_df[metrics_df["has_crack_gt"] == 1]
    n_crack = len(crack_only)
    n_nocrack = n_total - n_crack

    fp_no_crack = int(((metrics_df["has_crack_gt"] == 0) & (metrics_df["pred_any"] == 1)).sum())
    fp_rate = fp_no_crack / max(1, n_nocrack)

    summary = {
        "run_name": run_name,
        "img_size": img_size,
        "threshold": THR,
        "chosen_encoder": chosen_encoder,
        "checkpoint_path": MODEL1_CKPT,
        "device": str(device),
        "n_total": n_total,
        "n_crack": n_crack,
        "n_no_crack": n_nocrack,
        "mean_iou_all_empty_aware": float(metrics_df["iou_empty_aware"].mean()),
        "mean_iou_crack_only": float(crack_only["iou_empty_aware"].mean()) if n_crack else float("nan"),
        "mean_dice_crack_only": float(crack_only["dice_empty_aware"].mean()) if n_crack else float("nan"),
        "mean_pixel_acc_all": float(metrics_df["pixel_acc"].mean()),
        "no_crack_fp_count": fp_no_crack,
        "no_crack_fp_rate": float(fp_rate),
        "note": "Part 2 will generate PR curve, plots, overlays, galleries, detections jsonl."
    }

    pd.DataFrame([summary]).to_csv(out_dir / f"{run_name}_val_summary_report.csv", index=False)
    (out_dir / f"{run_name}_eval_config.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print(f" Finished eval {img_size}. Saved CSVs to: {out_dir}")
    return summary

# Cell 9 — Run both 256 and 448

In [9]:
summaries = []
for sz in EVAL_SIZES:
    summaries.append(eval_one_size(sz))

print("\n All done. Summaries:")
for s in summaries:
    print(s["run_name"], "| iou_all:", round(s["mean_iou_all_empty_aware"], 4), "| crack_iou:", round(s["mean_iou_crack_only"], 4))

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

 Finished eval 256. Saved CSVs to: /kaggle/working/output/phase7_model1/unet_efficientnetb3_eval256
 Finished eval 448. Saved CSVs to: /kaggle/working/output/phase7_model1/unet_efficientnetb3_eval448

 All done. Summaries:
unet_efficientnetb3_eval256 | iou_all: 0.5007 | crack_iou: 0.4297
unet_efficientnetb3_eval448 | iou_all: 0.6049 | crack_iou: 0.552


# Cell 10 — Install/check extra libs (only if missing)

In [10]:

import sys, subprocess

def ensure(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", pkg])

try:
    import matplotlib.pyplot as plt
except Exception:
    ensure("matplotlib")
    import matplotlib.pyplot as plt

try:
    from sklearn.metrics import precision_recall_curve, auc
except Exception:
    ensure("scikit-learn")
    from sklearn.metrics import precision_recall_curve, auc

print(" matplotlib + sklearn ready")

 matplotlib + sklearn ready


# Cell 11 — Helper: load run folders + metrics CSV

In [11]:
import pandas as pd
from pathlib import Path
import glob, os, json

OUT_ROOT = Path("/kaggle/working/output/phase7_model1")

RUN_DIRS = {
    256: OUT_ROOT / "unet_efficientnetb3_eval256",
    448: OUT_ROOT / "unet_efficientnetb3_eval448",
}

for sz, d in RUN_DIRS.items():
    assert d.exists(), f"Missing run folder: {d}"
    print(sz, "->", d)

def load_metrics(sz):
    d = RUN_DIRS[sz]
    metrics_path = list(d.glob("*_val_metrics.csv"))[0]
    summary_path = list(d.glob("*_val_summary_report.csv"))[0]
    mdf = pd.read_csv(metrics_path)
    sdf = pd.read_csv(summary_path)
    return d, mdf, sdf

print(" Run folders found")

256 -> /kaggle/working/output/phase7_model1/unet_efficientnetb3_eval256
448 -> /kaggle/working/output/phase7_model1/unet_efficientnetb3_eval448
 Run folders found


# Cell 12 — Plot histograms (IoU + confidence + crack area)

In [12]:
import matplotlib.pyplot as plt
import numpy as np

def save_hist(values, title, xlabel, out_path, bins=30):
    plt.figure(figsize=(7,4))
    plt.hist(values, bins=bins)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

def make_hist_bundle(sz):
    out_dir, mdf, _ = load_metrics(sz)

    # IoU hist (all)
    save_hist(
        mdf["iou_empty_aware"].values,
        f"Validation IoU Distribution (All) — eval{sz}",
        "IoU",
        out_dir / "unet_efficientnetb3_val_iou_hist_all.png"
    )

    # IoU hist (crack-only)
    crack = mdf[mdf["has_crack_gt"] == 1]
    save_hist(
        crack["iou_empty_aware"].values,
        f"Validation IoU Distribution (Crack-present only) — eval{sz}",
        "IoU",
        out_dir / "unet_efficientnetb3_val_iou_hist_crack_only.png"
    )

    # confidence mean pred
    save_hist(
        mdf["conf_mean_pred"].values,
        f"Confidence (mean prob inside predicted mask) — eval{sz}",
        "conf_mean_pred",
        out_dir / "unet_efficientnetb3_confidence_mean_pred_hist.png"
    )

    # confidence p95 pred
    save_hist(
        mdf["conf_p95_pred"].values,
        f"Confidence (P95 prob inside predicted mask) — eval{sz}",
        "conf_p95_pred",
        out_dir / "unet_efficientnetb3_confidence_p95_hist.png"
    )

    # GT crack area
    save_hist(
        crack["gt_pixels"].values,
        f"GT Crack Area Proxy (pixels) — Crack-present only — eval{sz}",
        "GT crack pixels",
        out_dir / "unet_efficientnetb3_gt_crack_area_hist.png"
    )

    # Pred crack area (all)
    save_hist(
        mdf["pred_pixels"].values,
        f"Predicted Crack Area Proxy (pixels) — All val — eval{sz}",
        "Pred crack pixels",
        out_dir / "unet_efficientnetb3_pred_crack_area_hist.png"
    )

    print(f" Saved histogram bundle for eval{sz} in {out_dir}")

for sz in [256, 448]:
    make_hist_bundle(sz)

 Saved histogram bundle for eval256 in /kaggle/working/output/phase7_model1/unet_efficientnetb3_eval256
 Saved histogram bundle for eval448 in /kaggle/working/output/phase7_model1/unet_efficientnetb3_eval448


# Cell 13 — Pixel PR curve + AUC

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc

PIXEL_SAMPLE_BUDGET = 300_000  # same style as your Model-3

def compute_pixel_pr(sz):
    out_dir = RUN_DIRS[sz]
    preprocess_fn = smp.encoders.get_preprocessing_fn(chosen_encoder, pretrained="imagenet")

    ds = CrackValDataset(val_df, img_size=sz, preprocess_fn=preprocess_fn)
    dl = DataLoader(ds, batch_size=4, shuffle=False, num_workers=2)

    rng = np.random.default_rng(SEED)
    y_true = []
    y_score = []

    for imgs, msks, _ in dl:
        imgs = imgs.to(device)
        with torch.inference_mode():
            prob = torch.sigmoid(model(imgs)).cpu().numpy()[:,0]   # (B,H,W)

        gt = msks.numpy()[:,0].astype(np.uint8)

        for b in range(prob.shape[0]):
            flat_prob = prob[b].reshape(-1)
            flat_gt   = gt[b].reshape(-1)

            remaining = PIXEL_SAMPLE_BUDGET - len(y_true)
            if remaining <= 0:
                break

            take = min(remaining, flat_prob.size)
            idx = rng.choice(flat_prob.size, size=take, replace=False)

            y_true.extend(flat_gt[idx].tolist())
            y_score.extend(flat_prob[idx].tolist())

        if len(y_true) >= PIXEL_SAMPLE_BUDGET:
            break

    y_true = np.array(y_true, dtype=np.uint8)
    y_score = np.array(y_score, dtype=np.float32)

    prec, rec, _ = precision_recall_curve(y_true, y_score)
    pr_auc = auc(rec, prec)

    pr_df = pd.DataFrame({"recall": rec, "precision": prec})
    pr_df.to_csv(out_dir / "unet_efficientnetb3_val_pr_auc_pixel.csv", index=False)

    plt.figure(figsize=(7,4))
    plt.plot(rec, prec)
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"Pixel Precision–Recall (AUC={pr_auc:.3f}, sampled={len(y_true)}) — eval{sz}")
    plt.tight_layout()
    plt.savefig(out_dir / "unet_efficientnetb3_pixel_pr_curve.png", dpi=200)
    plt.close()

    print(f" PR-AUC eval{sz}: {pr_auc:.4f}")
    return float(pr_auc)

pr256 = compute_pixel_pr(256)
pr448 = compute_pixel_pr(448)

 PR-AUC eval256: 0.7053
 PR-AUC eval448: 0.9159


# Cell 14 — Detections JSONL from masks (AABB + OBB)

In [14]:
MIN_COMPONENT_AREA = 30
print("MIN_COMPONENT_AREA =", MIN_COMPONENT_AREA)

MIN_COMPONENT_AREA = 30


In [15]:
import json, numpy as np
import cv2
from pathlib import Path

def components_to_boxes(bin_mask, min_area=30):
    # connected components
    n, labels, stats, _ = cv2.connectedComponentsWithStats(bin_mask.astype(np.uint8), connectivity=8)
    boxes = []
    H, W = bin_mask.shape

    for k in range(1, n):
        x, y, w, h, area = stats[k]
        if area < min_area:
            continue

        comp = (labels == k).astype(np.uint8)
        contours, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        cnt = max(contours, key=cv2.contourArea)

        rect = cv2.minAreaRect(cnt)                 # ((cx,cy),(rw,rh),angle)
        pts = cv2.boxPoints(rect)                   # 4x2
        pts = np.clip(pts, [0,0], [W-1, H-1]).tolist()

        boxes.append({
            "aabb": [int(x), int(y), int(x+w), int(y+h)],
            "obb": pts,
            "area": int(area),
        })
    return boxes

def write_detections(sz):
    out_dir = RUN_DIRS[sz]
    preprocess_fn = smp.encoders.get_preprocessing_fn(chosen_encoder, pretrained="imagenet")

    ds = CrackValDataset(val_df, img_size=sz, preprocess_fn=preprocess_fn)
    dl = DataLoader(ds, batch_size=1, shuffle=False, num_workers=2)

    out_path = out_dir / "unet_efficientnetb3_val_detections_from_masks.jsonl"
    f = open(out_path, "w", encoding="utf-8")

    for imgs, msks, metas in dl:
        imgs = imgs.to(device)
        with torch.inference_mode():
            prob = torch.sigmoid(model(imgs)).cpu().numpy()[0,0]
        pr = (prob >= THR).astype(np.uint8)

        gt = msks.numpy()[0,0].astype(np.uint8)

        gt_boxes = components_to_boxes(gt, min_area=MIN_COMPONENT_AREA)
        pr_boxes = components_to_boxes(pr, min_area=MIN_COMPONENT_AREA)

        # component confidence: mean prob inside each predicted component bbox (simple and stable)
        # (optional) you can upgrade later to mean prob inside component pixels
        record = {
            "stem": metas["stem"][0],
            "source": metas["source"][0],
            "image_path": metas["image_path"][0],
            "mask_path": metas["mask_path"][0],
            "img_size": sz,
            "thr": THR,
            "min_component_area": MIN_COMPONENT_AREA,
            "gt_boxes": gt_boxes,
            "pred_boxes": pr_boxes,
        }
        f.write(json.dumps(record) + "\n")

    f.close()
    print(" Wrote detections:", out_path)

write_detections(256)
write_detections(448)

 Wrote detections: /kaggle/working/output/phase7_model1/unet_efficientnetb3_eval256/unet_efficientnetb3_val_detections_from_masks.jsonl
 Wrote detections: /kaggle/working/output/phase7_model1/unet_efficientnetb3_eval448/unet_efficientnetb3_val_detections_from_masks.jsonl


# Cell 15 — Quick “BBox + OBB overlays” montage (10 samples)



In [16]:
import matplotlib.pyplot as plt
import numpy as np
import cv2

def draw_boxes(img, boxes, color=(0,255,0)):
    img2 = img.copy()
    for b in boxes:
        x1,y1,x2,y2 = b["aabb"]
        cv2.rectangle(img2, (x1,y1), (x2,y2), color, 2)
        pts = np.array(b["obb"], dtype=np.int32)
        cv2.polylines(img2, [pts], isClosed=True, color=color, thickness=2)
    return img2

def overlay_mask(img, mask, color=(255,0,0), alpha=0.5):
    out = img.copy().astype(np.float32)
    m = mask.astype(bool)
    out[m] = out[m]*(1-alpha) + np.array(color, dtype=np.float32)*alpha
    return out.astype(np.uint8)

def make_overlay_montage(sz, n_samples=10):
    out_dir = RUN_DIRS[sz]
    preprocess_fn = smp.encoders.get_preprocessing_fn(chosen_encoder, pretrained="imagenet")

    # sample indices (mix crack/no-crack)
    mdf = pd.read_csv(list(out_dir.glob("*_val_metrics.csv"))[0])
    idxs = list(np.linspace(0, len(mdf)-1, n_samples, dtype=int))

    fig, axes = plt.subplots(n_samples, 4, figsize=(12, 3*n_samples))
    if n_samples == 1:
        axes = np.expand_dims(axes, 0)

    for r, i in enumerate(idxs):
        row = mdf.iloc[i]
        img_path = row["image_path"]
        msk_path = row["mask_path"]

        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (sz, sz), interpolation=cv2.INTER_AREA)
        gt = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
        gt = cv2.resize(gt, (sz, sz), interpolation=cv2.INTER_NEAREST)
        gt = (gt > 127).astype(np.uint8)

        # predict
        img_in = img.astype(np.float32)
        img_in = preprocess_fn(img_in)
        x = torch.from_numpy(img_in).permute(2,0,1).unsqueeze(0).float().to(device)

        with torch.inference_mode():
            prob = torch.sigmoid(model(x)).cpu().numpy()[0,0]
        pr = (prob >= THR).astype(np.uint8)

        gt_boxes = components_to_boxes(gt, min_area=MIN_COMPONENT_AREA)
        pr_boxes = components_to_boxes(pr, min_area=MIN_COMPONENT_AREA)

        gt_vis = draw_boxes(img, gt_boxes, color=(0,255,0))
        pr_vis = draw_boxes(img, pr_boxes, color=(255,0,0))
        ov_vis = overlay_mask(img, pr, color=(255,0,0), alpha=0.45)

        for c, (title, im) in enumerate([
            ("Image", img),
            ("GT AABB+OBB", gt_vis),
            ("Pred AABB+OBB", pr_vis),
            ("Pred overlay", ov_vis),
        ]):
            ax = axes[r, c]
            ax.imshow(im)
            ax.set_title(title, fontsize=10)
            ax.axis("off")

    plt.tight_layout()
    out_path = out_dir / "unet_efficientnetb3_bbox_obb_overlays.png"
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(" Saved montage:", out_path)

make_overlay_montage(448, n_samples=10)   # main parity size

 Saved montage: /kaggle/working/output/phase7_model1/unet_efficientnetb3_eval448/unet_efficientnetb3_bbox_obb_overlays.png


# Cell 16 — Top-K worst galleries (crack-present only)

In [17]:
# Cell 16 — Top-K worst crack-only gallery
import matplotlib.pyplot as plt
import numpy as np
import cv2

def make_topk_worst_crack_only(sz, k=12):
    out_dir = RUN_DIRS[sz]
    preprocess_fn = smp.encoders.get_preprocessing_fn(chosen_encoder, pretrained="imagenet")

    mdf = pd.read_csv(list(out_dir.glob("*_val_metrics.csv"))[0])
    crack = mdf[mdf["has_crack_gt"] == 1].sort_values("iou_empty_aware", ascending=True).head(k)

    fig, axes = plt.subplots(k, 5, figsize=(15, 3*k))
    if k == 1:
        axes = np.expand_dims(axes, 0)

    for r, (_, row) in enumerate(crack.iterrows()):
        img_path = row["image_path"]
        msk_path = row["mask_path"]

        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (sz, sz), interpolation=cv2.INTER_AREA)

        gt = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
        gt = cv2.resize(gt, (sz, sz), interpolation=cv2.INTER_NEAREST)
        gt = (gt > 127).astype(np.uint8)

        img_in = preprocess_fn(img.astype(np.float32))
        x = torch.from_numpy(img_in).permute(2,0,1).unsqueeze(0).float().to(device)

        with torch.inference_mode():
            prob = torch.sigmoid(model(x)).cpu().numpy()[0,0]
        pr = (prob >= THR).astype(np.uint8)

        overlay = overlay_mask(img, pr, color=(255,0,0), alpha=0.45)

        # prob heatmap
        heat = (prob * 255).clip(0,255).astype(np.uint8)
        heat = cv2.applyColorMap(heat, cv2.COLORMAP_MAGMA)
        heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)

        panels = [
            ("Image", img),
            ("GT", gt*255),
            (f"Pred (IoU={row['iou_empty_aware']:.3f})", pr*255),
            ("Overlay", overlay),
            ("Prob", heat),
        ]

        for c,(title,im) in enumerate(panels):
            ax = axes[r,c]
            ax.imshow(im, cmap="gray" if im.ndim==2 else None)
            ax.set_title(title, fontsize=9)
            ax.axis("off")

    plt.tight_layout()
    out_path = out_dir / "unet_efficientnetb3_topk_worst_crack_only_gallery.png"
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(" Saved worst gallery:", out_path)

make_topk_worst_crack_only(448, k=12)

 Saved worst gallery: /kaggle/working/output/phase7_model1/unet_efficientnetb3_eval448/unet_efficientnetb3_topk_worst_crack_only_gallery.png


# Cell 17 — Zip the eval448 folder

In [18]:
import shutil
from pathlib import Path

def zip_run_safe(sz):
    run_dir = RUN_DIRS[sz]                         # folder to zip
    zip_base = OUT_ROOT / f"BUNDLE_unet_efficientnetb3_eval{sz}"   # OUTSIDE run_dir

    # remove old zip if it exists
    old_zip = Path(str(zip_base) + ".zip")
    if old_zip.exists():
        old_zip.unlink()

    out_zip = shutil.make_archive(
        base_name=str(zip_base),   
        format='zip',
        root_dir=str(run_dir)      # zip contents from this folder
    )

    print(" Created zip:", out_zip)
    print("Zip size (MB):", round(Path(out_zip).stat().st_size / (1024 * 1024), 2))

zip_run_safe(448)

 Created zip: /kaggle/working/output/phase7_model1/BUNDLE_unet_efficientnetb3_eval448.zip
Zip size (MB): 14.31
